# 03 — Two-Stage Default · Combine (M × N grid + Position Weighted Avg)

CLF prob × REG pred → 모든 (clf, reg) 조합 unit pred. 4 clf × 5 reg = **20 그리드 조합**.

- **Step 1**: die-level `final_die = clf_prob × reg_pred`
- **Step 2 (a)**: die→unit mean → `oof|val|test_unit.csv`
- **Step 2 (b)**: position weighted avg (Optuna 50 trial × 그리드) → `oof|val|test_unit_weighted.csv`
- **출력**: `4_output/03_two_stage/default/combined/{clf}_x_{reg}/...` + `grid_summary.csv` + `weighted_summary.csv` + `combine_meta.json`


## 1. 환경 + 모듈 자동 탐지

In [ ]:
import os, sys, json

try:
    import google.colab
    %run /content/project/setup.py
except ImportError:
    %run ../../../setup.py

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from utils.config import PROJECT_ROOT, SEED, TARGET_COL, KEY_COL, DIE_KEY_COL, OUTPUT_DIR
from utils.data import load_all
from sklearn.model_selection import KFold

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

GRID_ROOT = os.path.join(OUTPUT_DIR, '03_two_stage', 'default')
CLF_DIR   = os.path.join(GRID_ROOT, 'clf')
REG_DIR   = os.path.join(GRID_ROOT, 'reg')
OUT_DIR   = os.path.join(GRID_ROOT, 'combined')
os.makedirs(OUT_DIR, exist_ok=True)

N_TRIALS_POS = 1           # ★ position weighted 그리드별 Optuna 예산
TIMEOUT_SEC  = None          # ★ §25

REQ_DIE = ['oof_die.csv', 'val_die.csv', 'test_die.csv']

def _list_models(root):
    if not os.path.exists(root): return {}
    out = {}
    for name in sorted(os.listdir(root)):
        path = os.path.join(root, name)
        if os.path.isdir(path) and all(os.path.exists(os.path.join(path, f)) for f in REQ_DIE):
            out[name] = path
    return out

clf_pool = _list_models(CLF_DIR)
reg_pool = _list_models(REG_DIR)
print(f'CLF 가용 ({len(clf_pool)}): {list(clf_pool)}')
print(f'REG 가용 ({len(reg_pool)}): {list(reg_pool)}')
print(f'Grid 조합 수: {len(clf_pool) * len(reg_pool)}')

if not clf_pool: raise RuntimeError(f'CLF 산출물 없음: {CLF_DIR}')
if not reg_pool: raise RuntimeError(f'REG 산출물 없음: {REG_DIR}')


## 2. CLF/REG die-level 로드 + 정합 검증

In [ ]:
_, ys = load_all()
y_train = ys['train'].set_index(KEY_COL)[TARGET_COL]
y_val   = ys['validation'].set_index(KEY_COL)[TARGET_COL]
y_test  = ys['test'].set_index(KEY_COL)[TARGET_COL]

def _load_die(path, split, value_col):
    df = pd.read_csv(os.path.join(path, f'{split}_die.csv'))
    return df[[KEY_COL, DIE_KEY_COL, value_col]].rename(columns={value_col: 'v'})

clf_die = {n: {sp: _load_die(p, sp, 'prob') for sp in ['oof','val','test']} for n, p in clf_pool.items()}
reg_die = {n: {sp: _load_die(p, sp, 'pred') for sp in ['oof','val','test']} for n, p in reg_pool.items()}

ref_die = {}
for sp in ['oof', 'val', 'test']:
    ref = next(iter(clf_die.values()))[sp][[KEY_COL, DIE_KEY_COL]]
    ref_die[sp] = ref
    for name, d in {**clf_die, **reg_die}.items():
        cur = d[sp][[KEY_COL, DIE_KEY_COL]]
        if len(cur) != len(ref):
            raise ValueError(f'{sp}/{name}: row 수 {len(cur)} != ref {len(ref)}')
        if not (cur.values == ref.values).all():
            raise ValueError(f'{sp}/{name}: (KEY, DIE_KEY) 순서 불일치')

print('[정합 OK] 모든 clf/reg die-level csv 동일 (KEY, DIE_KEY) 순서')


## 3. M × N grid 곱셈 + unit mean + RMSE

In [ ]:
def _unit_mean(die_df, value_col='v'):
    return die_df.groupby(KEY_COL, sort=False)[value_col].mean()

def _rmse(p, y):
    p = p.loc[y.index]
    return float(np.sqrt(np.mean((p.values - y.values) ** 2)))

summary_rows = []
combined_unit = {}

for c_name in clf_pool:
    for r_name in reg_pool:
        unit_preds = {}
        for sp, y_true in [('oof', y_train), ('val', y_val), ('test', y_test)]:
            cdf = clf_die[c_name][sp]
            rdf = reg_die[r_name][sp]
            final_die = cdf['v'].values * rdf['v'].values
            tmp = pd.DataFrame({KEY_COL: cdf[KEY_COL].values, 'v': final_die})
            unit_preds[sp] = _unit_mean(tmp)

        combined_unit[(c_name, r_name)] = unit_preds
        summary_rows.append({
            'clf': c_name, 'reg': r_name,
            'oof':  _rmse(unit_preds['oof'],  y_train),
            'val':  _rmse(unit_preds['val'],  y_val),
            'test': _rmse(unit_preds['test'], y_test),
        })

summary = pd.DataFrame(summary_rows).sort_values('val').reset_index(drop=True)
print('=== M × N Grid 결과 (val 오름차순) ===')
print(summary.to_string(index=False, float_format='%.6f'))

best_row = summary.iloc[0]
print(f'\nGrid best: {best_row["clf"]} × {best_row["reg"]} → val={best_row["val"]:.6f} test={best_row["test"]:.6f}')


## 3.5 분류 threshold τ 탐색 (strategy_common.md §9 / §18 매트릭스 APPLY)

각 (clf × reg) 조합마다 **die-level prob에 임계값 τ를 적용**하여 `prob < τ`이면 final=0으로 강제.

- τ ∈ [0.00, 0.05, 0.10, ..., 0.50] 그리드 (총 11개)
- **train OOF에서 best τ 탐색 → val 적용 → val 개선 시 채택** (§19 원칙)
- 채택되면 `combined_unit[(c, r)]`을 τ-적용본으로 덮어씀 → 이후 save/position weighting 이 τ-적용 결과 사용
- 1차 분석 (Recall 0.011) 컨텍스트에서 분류 무력화 보정 목적

In [ ]:
# τ 탐색 그리드 — die-level prob에 적용 (strategy_common §9)
TAU_GRID = np.arange(0.00, 0.55, 0.05)   # 0.0 ~ 0.50 step 0.05 (총 11개)

def _apply_tau(cdf, rdf, tau):
    # die-level final = clf_prob * reg_pred * (clf_prob >= tau). 결과 unit-mean.
    prob = cdf['v'].values
    pred = cdf['v'].values * rdf['v'].values
    pred = np.where(prob >= tau, pred, 0.0)
    tmp = pd.DataFrame({KEY_COL: cdf[KEY_COL].values, 'v': pred})
    return _unit_mean(tmp)

tau_decisions = []   # 조합별 결정 기록
for c_name in clf_pool:
    for r_name in reg_pool:
        cdf_oof = clf_die[c_name]['oof']
        rdf_oof = reg_die[r_name]['oof']
        # τ=0 baseline (이미 combined_unit에 있음)
        baseline = combined_unit[(c_name, r_name)]
        baseline_oof_rmse = _rmse(baseline['oof'], y_train)
        baseline_val_rmse = _rmse(baseline['val'], y_val)

        # 1) train OOF 에서 best τ
        best_tau, best_train_rmse = 0.0, baseline_oof_rmse
        for tau in TAU_GRID:
            if tau == 0.0:
                continue   # baseline 동일
            cand = _apply_tau(cdf_oof, rdf_oof, tau)
            r = _rmse(cand, y_train)
            if r < best_train_rmse:
                best_train_rmse, best_tau = r, float(tau)

        if best_tau == 0.0:
            tau_decisions.append({'clf': c_name, 'reg': r_name,
                                  'best_tau': 0.0, 'accepted': False,
                                  'reason': 'train OOF에서 개선 없음'})
            continue

        # 2) val 적용 → 개선 확인
        cand_val_unit  = _apply_tau(clf_die[c_name]['val'],  reg_die[r_name]['val'],  best_tau)
        cand_test_unit = _apply_tau(clf_die[c_name]['test'], reg_die[r_name]['test'], best_tau)
        cand_train_unit = _apply_tau(cdf_oof, rdf_oof, best_tau)
        cand_val_rmse  = _rmse(cand_val_unit,  y_val)

        if cand_val_rmse + 1e-9 < baseline_val_rmse:
            # 채택 — combined_unit 덮어쓰기
            combined_unit[(c_name, r_name)] = {
                'oof':  cand_train_unit,
                'val':  cand_val_unit,
                'test': cand_test_unit,
            }
            tau_decisions.append({
                'clf': c_name, 'reg': r_name,
                'best_tau': best_tau, 'accepted': True,
                'baseline_val': baseline_val_rmse, 'tuned_val': cand_val_rmse,
                'delta_val': cand_val_rmse - baseline_val_rmse,
            })
        else:
            tau_decisions.append({
                'clf': c_name, 'reg': r_name,
                'best_tau': best_tau, 'accepted': False,
                'baseline_val': baseline_val_rmse, 'tuned_val': cand_val_rmse,
                'reason': 'val 개선 없음 → 미적용',
            })

tau_df = pd.DataFrame(tau_decisions)
print('=== τ 탐색 결과 ===')
print(tau_df.to_string(index=False, float_format='%.6f'))

# summary 갱신 (τ 적용된 RMSE 반영)
summary_rows = []
for c_name in clf_pool:
    for r_name in reg_pool:
        u = combined_unit[(c_name, r_name)]
        summary_rows.append({
            'clf': c_name, 'reg': r_name,
            'oof':  _rmse(u['oof'],  y_train),
            'val':  _rmse(u['val'],  y_val),
            'test': _rmse(u['test'], y_test),
        })
summary = pd.DataFrame(summary_rows).sort_values('val').reset_index(drop=True)
print('\n=== τ 적용 후 M × N Grid (val 오름차순) ===')
print(summary.to_string(index=False, float_format='%.6f'))
best_row = summary.iloc[0]
print(f'\nGrid best: {best_row["clf"]} × {best_row["reg"]} → val={best_row["val"]:.6f} test={best_row["test"]:.6f}')


## 4. 모든 조합 unit csv 저장 (mean baseline)

In [ ]:
for (c_name, r_name), preds in combined_unit.items():
    sub = os.path.join(OUT_DIR, f'{c_name}_x_{r_name}')
    os.makedirs(sub, exist_ok=True)
    for sp, y_true in [('oof', y_train), ('val', y_val), ('test', y_test)]:
        s = preds[sp]
        df = pd.DataFrame({
            KEY_COL: s.index.values,
            'pred':  s.values,
            'health': y_true.reindex(s.index).values,
        })
        df.to_csv(os.path.join(sub, f'{sp}_unit.csv'), index=False)

summary.to_csv(os.path.join(OUT_DIR, 'grid_summary.csv'), index=False)
print(f'저장 완료: {OUT_DIR}')
print(f'  {len(combined_unit)}개 조합 × 3 split = {len(combined_unit)*3}개 csv')


## 5. Position weighted avg (그리드별 Optuna)

각 (clf, reg) 그리드 조합마다 die→unit 집계를 mean 대신 position 가중평균. Optuna가 4 weight (Dirichlet 정규화) 탐색.

In [ ]:
xs_full, _ = load_all()
pos_map = xs_full.set_index(DIE_KEY_COL)['position']

def _add_pos(ref_df):
    df = ref_df.copy()
    df['position'] = df[DIE_KEY_COL].map(pos_map).astype(int)
    return df

ref_oof_pos  = _add_pos(ref_die['oof'])
ref_val_pos  = _add_pos(ref_die['val'])
ref_test_pos = _add_pos(ref_die['test'])

def _pivot_die(uid, pos, vals):
    df = pd.DataFrame({KEY_COL: uid, 'pos': pos, 'v': vals})
    return df.pivot(index=KEY_COL, columns='pos', values='v')

weighted_results = {}

print('=== Position weighted avg (그리드별 Optuna) ===\n')
for c_name in clf_pool:
    for r_name in reg_pool:
        final_oof  = clf_die[c_name]['oof']['v'].values  * reg_die[r_name]['oof']['v'].values
        final_val  = clf_die[c_name]['val']['v'].values  * reg_die[r_name]['val']['v'].values
        final_test = clf_die[c_name]['test']['v'].values * reg_die[r_name]['test']['v'].values

        pivot_oof  = _pivot_die(ref_oof_pos[KEY_COL].values,  ref_oof_pos['position'].values,  final_oof)
        pivot_val  = _pivot_die(ref_val_pos[KEY_COL].values,  ref_val_pos['position'].values,  final_val)
        pivot_test = _pivot_die(ref_test_pos[KEY_COL].values, ref_test_pos['position'].values, final_test)

        baseline_oof  = float(np.sqrt(np.mean((pivot_oof.mean(axis=1).reindex(y_train.index).values  - y_train.values)**2)))
        baseline_val  = float(np.sqrt(np.mean((pivot_val.mean(axis=1).reindex(y_val.index).values    - y_val.values)**2)))
        baseline_test = float(np.sqrt(np.mean((pivot_test.mean(axis=1).reindex(y_test.index).values  - y_test.values)**2)))

        piv_oof_arr  = pivot_oof.reindex(y_train.index).values
        piv_val_arr  = pivot_val.reindex(y_val.index).values
        piv_test_arr = pivot_test.reindex(y_test.index).values
        y_tr_arr, y_vl_arr, y_te_arr = y_train.values, y_val.values, y_test.values

        def objective(trial, _po=piv_oof_arr, _yo=y_tr_arr):
            w_raw = np.array([trial.suggest_float(f'w{p}', 0.05, 1.0) for p in [1,2,3,4]])
            w = w_raw / w_raw.sum()
            return float(np.sqrt(np.mean(((_po * w).sum(axis=1) - _yo)**2)))

        study = optuna.create_study(direction='minimize',
                                    sampler=optuna.samplers.TPESampler(seed=SEED))
        study.optimize(objective, n_trials=N_TRIALS_POS, timeout=TIMEOUT_SEC, show_progress_bar=False)

        bw = np.array([study.best_trial.params[f'w{p}'] for p in [1,2,3,4]])
        bw = bw / bw.sum()

        w_oof  = (piv_oof_arr  * bw).sum(axis=1)
        w_val  = (piv_val_arr  * bw).sum(axis=1)
        w_test = (piv_test_arr * bw).sum(axis=1)

        weighted_results[(c_name, r_name)] = {
            'baseline':     {'oof': baseline_oof, 'val': baseline_val, 'test': baseline_test},
            'weighted':     {'oof': float(np.sqrt(np.mean((w_oof  - y_tr_arr)**2))),
                             'val': float(np.sqrt(np.mean((w_val  - y_vl_arr)**2))),
                             'test':float(np.sqrt(np.mean((w_test - y_te_arr)**2)))},
            'best_weights': bw.tolist(),
            'pred':         {'oof': pd.Series(w_oof,  index=y_train.index),
                             'val': pd.Series(w_val,  index=y_val.index),
                             'test':pd.Series(w_test, index=y_test.index)},
        }

        wo = weighted_results[(c_name, r_name)]['weighted']
        ba = weighted_results[(c_name, r_name)]['baseline']
        print(f'  [{c_name} × {r_name}] mean: oof={ba["oof"]:.6f} val={ba["val"]:.6f}  weighted: val={wo["val"]:.6f} (Δ={wo["val"]-ba["val"]:+.6f})')

# weighted summary
wr_rows = []
for (c, r), data in weighted_results.items():
    bw_ = data['best_weights']
    wr_rows.append({
        'clf': c, 'reg': r,
        'mean_oof':  data['baseline']['oof'],   'mean_val':  data['baseline']['val'],   'mean_test':  data['baseline']['test'],
        'weighted_oof': data['weighted']['oof'], 'weighted_val': data['weighted']['val'], 'weighted_test': data['weighted']['test'],
        'delta_val':  data['weighted']['val']  - data['baseline']['val'],
        'delta_test': data['weighted']['test'] - data['baseline']['test'],
        'w1': bw_[0], 'w2': bw_[1], 'w3': bw_[2], 'w4': bw_[3],
    })
weighted_summary = pd.DataFrame(wr_rows).sort_values('weighted_val').reset_index(drop=True)
weighted_summary.to_csv(os.path.join(OUT_DIR, 'weighted_summary.csv'), index=False)
print('\n=== Weighted summary (val 오름차순) ===')
print(weighted_summary.to_string(index=False, float_format='%.6f'))

# weighted unit csv 저장
for (c, r), data in weighted_results.items():
    sub = os.path.join(OUT_DIR, f'{c}_x_{r}')
    os.makedirs(sub, exist_ok=True)
    for sp_key, y_ref in [('oof', y_train), ('val', y_val), ('test', y_test)]:
        s = data['pred'][sp_key]
        df = pd.DataFrame({
            KEY_COL: s.index.values,
            'pred':  s.values,
            'health': y_ref.reindex(s.index).values,
        })
        df.to_csv(os.path.join(sub, f'{sp_key}_unit_weighted.csv'), index=False)
print(f'\nweighted unit csv 저장 완료')


## 6. 메타 저장

In [ ]:
meta = {
    'clf_pool':  list(clf_pool),
    'reg_pool':  list(reg_pool),
    'n_grid_combinations': len(combined_unit),
    'grid_summary_top10': summary.head(10).to_dict(orient='records'),
    'grid_best_mean': {
        'clf': str(best_row['clf']), 'reg': str(best_row['reg']),
        'oof': float(best_row['oof']), 'val': float(best_row['val']), 'test': float(best_row['test']),
    },
    'weighted_grid': {
        'n_trials_per_study': N_TRIALS_POS,
        'timeout_sec':        TIMEOUT_SEC,
        'n_studies':          len(weighted_results),
        'summary_sorted':     weighted_summary.to_dict(orient='records'),
        'best_weighted_val':  float(weighted_summary.iloc[0]['weighted_val']),
        'best_combo':         f'{weighted_summary.iloc[0]["clf"]} × {weighted_summary.iloc[0]["reg"]}',
    },
}
with open(os.path.join(OUT_DIR, 'combine_meta.json'), 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False, default=str)

for f_ in sorted(os.listdir(OUT_DIR)):
    p = os.path.join(OUT_DIR, f_)
    if os.path.isfile(p):
        sz = os.path.getsize(p) / 1024
        print(f'  {f_:30s}  {sz:>10,.1f} KB')
